In [ ]:
# ════════════════════════════════════════════════════════════
#  STEP 2: Generate SQL from pred_plan → compute Token F1
#  Input : dev_150_with_pred_plans.json  +  lora_adapter_plan2sql.zip
#  Output: dev_500_final_results.json
# ════════════════════════════════════════════════════════════

!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

import json, zipfile, os, re, gc, time
from collections import Counter
import torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel

# ── 1. Unzip plan2sql adapter ─────────────────────────────────
ADAPTER_ZIP  = "./lora_adapter_plan2sql (1).zip"
ADAPTER_DIR  = "./lora_adapter_plan2sql"
INPUT_PATH   = "./dev_500_with_pred_plans.json"
OUTPUT_PATH  = "./dev_500_final_results.json"
MAX_INPUT    = 384
MAX_TARGET   = 384

if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(ADAPTER_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted to:", ADAPTER_DIR)

# ── 2. Load model ─────────────────────────────────────────────
MODEL_NAME = "google/flan-t5-xl"

dtype = torch.bfloat16 if torch.cuda.is_available() and \
        torch.cuda.get_device_capability()[0] >= 8 else torch.float16

os.makedirs("./offload", exist_ok=True)

print(f"Loading base model ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype    = dtype,
    device_map     = "auto",
    offload_folder = "./offload",
)

print("Loading plan2sql LoRA...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable   = False,
    offload_folder = "./offload",
)
model.eval()
model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print("Model ready!")

# ── 3. Load input file ────────────────────────────────────────
with open(INPUT_PATH, encoding="utf-8") as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples")

# ── 4. Generate SQL ───────────────────────────────────────────
def generate_sql(plan_text):
    inp = tokenizer(
        plan_text,
        return_tensors = "pt",
        max_length     = MAX_INPUT,
        truncation     = True,
    )
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens = MAX_TARGET,
            num_beams      = 4,
            early_stopping = True,
            length_penalty = 1.0,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

start = time.time()
for i, r in enumerate(data):
    r["pred_sql"] = generate_sql(r["pred_plan"])
    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        eta = elapsed / (i + 1) * (len(data) - i - 1)
        print(f"  [{i+1}/{len(data)}]  elapsed: {elapsed/60:.1f} min  ETA: {eta/60:.1f} min")

# ── 5. Token F1 ───────────────────────────────────────────────
def tokenize_sql(text):
    return re.findall(r'\w+', text.lower())

def token_f1(pred, gold):
    pred_tokens = tokenize_sql(pred)
    gold_tokens = tokenize_sql(gold)
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    pred_count = Counter(pred_tokens)
    gold_count = Counter(gold_tokens)
    overlap    = sum((pred_count & gold_count).values())
    precision  = overlap / len(pred_tokens)
    recall     = overlap / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

f1_scores = []
for r in data:
    f1 = token_f1(r["pred_sql"], r["gold_sql"])
    f1_scores.append(f1)
    r["sql_f1"] = round(f1, 4)

avg_f1 = sum(f1_scores) / len(f1_scores)

print("\n══════════════════════════════════════")
print(f"    SQL EVALUATION (n={len(data)})")
print("══════════════════════════════════════")
print(f"  Token F1 : {avg_f1*100:.2f}%")
print("══════════════════════════════════════")

# ── 6. Save ───────────────────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
print(f"\nSaved → {OUTPUT_PATH}")

# ── 7. Sample predictions ─────────────────────────────────────
print("\n── Sample predictions ──")
for i in [0, 10, 50]:
    if i < len(data):
        r = data[i]
        print(f"\n[{i}] SQL F1={r['sql_f1']:.3f}")
        print(f"  PRED PLAN : {r['pred_plan']}")
        print(f"  GOLD SQL  : {r['gold_sql']}")
        print(f"  PRED SQL  : {r['pred_sql']}")

Extracted to: ./lora_adapter_plan2sql
Loading base model (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading plan2sql LoRA...
Model ready!
Loaded 150 samples
  [25/150]  elapsed: 1.0 min  ETA: 4.9 min
  [50/150]  elapsed: 2.4 min  ETA: 4.7 min
  [75/150]  elapsed: 3.9 min  ETA: 3.9 min
  [100/150]  elapsed: 5.3 min  ETA: 2.6 min
  [125/150]  elapsed: 6.8 min  ETA: 1.4 min
  [150/150]  elapsed: 8.0 min  ETA: 0.0 min

══════════════════════════════════════
    SQL EVALUATION (n=150)
══════════════════════════════════════
  Token F1 : 94.80%
══════════════════════════════════════

Saved → ./dev_500_final_results.json

── Sample predictions ──

[0] SQL F1=1.000
  PRED PLAN : step1: SCAN | table: singer || step2: AGGREGATE | Scalar aggregate (no GROUP BY) -> compute COUNT(*) || step3: PROJECT | columns: COUNT(*)
  GOLD SQL  : SELECT count(*) FROM singer
  PRED SQL  : SELECT count(*) FROM singer

[10] SQL F1=1.000
  PRED PLAN : step1: SCAN | table: singer || step2: AGGREGATE | group_by: Country | compute: COUNT(*) || step3: PROJECT | columns: Country, COUNT(*)
  GOLD SQL  : SELECT country ,

In [4]:
from google.colab import files
uploaded =files.upload()

Saving lora_adapter_plan2sql (1).zip to lora_adapter_plan2sql (1).zip


In [5]:
from google.colab import files
uploaded =files.upload()

Saving dev_500_with_pred_plans.json to dev_500_with_pred_plans.json
